# Clustering Student Profiles

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA


In [ ]:
df = pd.read_csv('../data/processed/stream_mapped_career_dataset.csv')


In [ ]:
numeric_features = df.select_dtypes(include=['int64','float64']).columns.tolist()
X = df[numeric_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
results = []

for k in range(2,8):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    results.append((k, score))

results

In [ ]:
best_k = max(results, key=lambda x: x[1])[0]

kmeans_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)

df['student_cluster'] = kmeans_model.fit_predict(X_scaled)

df['student_cluster'].value_counts()

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=df['student_cluster'])
plt.title('Student Profile Clusters')
plt.show()

In [ ]:
df.to_csv('../data/processed/clustered_career_stream_dataset.csv', index=False)

joblib.dump(kmeans_model, '../models/kmeans_student_profile_model.pkl')
joblib.dump(scaler, '../models/cluster_scaler.pkl')

print('Saved clustering models')